<a href="https://colab.research.google.com/github/ruraaara/Lumen/blob/main/Dasboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install geopandas rapidfuzz -q

import os
import re
import json
import pandas as pd
import geopandas as gpd
from google.colab import drive
from rapidfuzz import process, fuzz

drive.mount('/content/drive')

MASTER_PATH = '/content/drive/MyDrive/ARSEN/Daftar_Kabupaten_Kota_Indonesia_2026.xlsx'
GADM_SHP_PATH = '/content/drive/MyDrive/ARSEN/GADm BatS/gadm41_IDN_2.shp'
OUTPUT_FOLDER = '/content/drive/MyDrive/ARSEN/hasil_standarisasi'
PANEL_CSV = f'{OUTPUT_FOLDER}/panel_kabkota_final.csv'
ZONAL_JSON = f'{OUTPUT_FOLDER}/data_dashboard_nightlight.json'
METHODS_JSON = f'{OUTPUT_FOLDER}/methods_summary.json'


master = pd.read_excel(MASTER_PATH)
master.columns = [c.strip() for c in master.columns]

master.loc[master['Nama Daerah'] == 'Pasangkayu', 'Jenis'] = 'KABUPATEN'
master.loc[master['Nama Daerah'] == 'Toba Samosir', 'Nama Daerah'] = 'Toba'
master = pd.concat([master, pd.DataFrame([
    {'Provinsi': 'Sumatera Utara', 'Nama Daerah': 'Sibolga', 'Jenis': 'KOTA'},
    {'Provinsi': 'Sulawesi Selatan', 'Nama Daerah': 'Sidenreng Rappang', 'Jenis': 'KABUPATEN'},
    {'Provinsi': 'Sulawesi Tenggara', 'Nama Daerah': 'Baubau', 'Jenis': 'KOTA'},
    {'Provinsi': 'Gorontalo', 'Nama Daerah': 'Pohuwato', 'Jenis': 'KABUPATEN'},
])], ignore_index=True)

master['nama_gabungan'] = master['Jenis'].str.title() + ' ' + master['Nama Daerah']
master_lookup = master['nama_gabungan'].tolist()

def compact(s):
    return re.sub(r'\s+', '', re.sub(r'^(Kabupaten|Kota)\s*', '', s, flags=re.IGNORECASE)).lower()

master_compact_map = {}
for m in master_lookup:
    master_compact_map.setdefault(compact(m), []).append(m)

KOREKSI_MANUAL = {
    'Kota Baru': 'Kabupaten Kotabaru', 'Kota Sampit': 'Kabupaten Kotawaringin Timur',
    'Kota Tanjung': 'Kabupaten Tabalong', 'Kota Watampone': 'Kabupaten Bone',
    'Kota Singaraja': 'Kabupaten Buleleng', 'Kota Tanjung Pandan': 'Kabupaten Belitung',
    'Kota Tembilahan': 'Kabupaten Indragiri Hilir', 'Kota Meulaboh': 'Kabupaten Aceh Barat',
    'Kota Maumere': 'Kabupaten Sikka', 'Kota Padang Sidempuan': 'Kota Padangsidimpuan',
    'Kota Surakarta (Solo)': 'Kota Surakarta', 'Pali': 'Kabupaten Penukal Abab Lematang Ilir',
    'Toba Samosir / Toba': 'Kabupaten Toba', 'Mamuju Utara': 'Kabupaten Pasangkayu',
    'Mamuju Utara / Pasangkayu': 'Kabupaten Pasangkayu',
    'Siau Tagulandang Biaro': 'Kabupaten Kepulauan Siau Tagulandang Biaro',
    'Pontianak': 'Kabupaten Mempawah', 'Tanjung Jabung B': 'Kabupaten Tanjung Jabung Barat',
    'Tanjung Jabung T': 'Kabupaten Tanjung Jabung Timur',
    'Maluku Tenggara Barat': 'Kabupaten Kepulauan Tanimbar',
}
KATA_NON_ADMINISTRATIF = ('danau ', 'waduk ')

def clean_name(name):
    name = str(name).strip()
    name = re.sub(r'\s+', ' ', name)
    name = re.sub(r'^Kab\.\s*', 'Kabupaten ', name, flags=re.IGNORECASE)
    name = re.sub(r'^Kab\s+', 'Kabupaten ', name, flags=re.IGNORECASE)
    return name.strip()

def match_kabkota(nama_asli):
    if any(k in str(nama_asli).lower() for k in KATA_NON_ADMINISTRATIF):
        return None
    if nama_asli in KOREKSI_MANUAL:
        return KOREKSI_MANUAL[nama_asli]
    nama_bersih = clean_name(nama_asli)
    sudah_ada_prefix = bool(re.match(r'^(Kabupaten|Kota)\s', nama_bersih, flags=re.IGNORECASE))
    kandidat_coba = [nama_bersih]
    if not sudah_ada_prefix:
        kandidat_coba.insert(0, f'Kabupaten {nama_bersih}')
    for coba in kandidat_coba:
        if coba in master_lookup:
            return coba
        kandidat_compact = master_compact_map.get(compact(coba), [])
        if len(kandidat_compact) == 1:
            return kandidat_compact[0]
    fz = process.extractOne(nama_bersih, master_lookup, scorer=fuzz.token_sort_ratio)
    return fz[0] if fz and fz[1] >= 90 else None


gdf = gpd.read_file(GADM_SHP_PATH)
gdf['nama_standar'] = gdf['NAME_2'].apply(match_kabkota)
print('geometri wilayah:', len(gdf), '| gagal dicocokkan:', gdf['nama_standar'].isna().sum())

panel = pd.read_csv(PANEL_CSV)
with open(ZONAL_JSON) as f:
    data_mismatch = json.load(f)
with open(METHODS_JSON) as f:
    methods = json.load(f)

print('panel_kabkota_final:', panel.shape)
print('wilayah di data_dashboard_nightlight.json:', len(data_mismatch))


mismatch_map = {d['id']: d for d in data_mismatch}

gdf_map = gdf[['nama_standar', 'geometry']].dropna(subset=['nama_standar']).drop_duplicates(subset=['nama_standar']).copy()
gdf_map['geometry'] = gdf_map.geometry.simplify(0.01, preserve_topology=True)
if gdf_map.crs is not None and gdf_map.crs.to_epsg() != 4326:
    gdf_map = gdf_map.to_crs(4326)

lookup_gk = dict(zip(panel['nama_standar'], panel['garis_kemiskinan']))
lookup_prov = dict(zip(panel['nama_standar'], panel['provinsi']))
kolom_sumber = 'garis_kemiskinan_sumber'
lookup_sumber = dict(zip(panel['nama_standar'], panel[kolom_sumber])) if kolom_sumber in panel.columns else {}

gdf_map['garis_kemiskinan'] = gdf_map['nama_standar'].map(lookup_gk)
gdf_map['provinsi'] = gdf_map['nama_standar'].map(lookup_prov)
gdf_map['sumber'] = gdf_map['nama_standar'].map(lookup_sumber).fillna('tidak diketahui')

# FIX: wilayah yang gak ada di mismatch_map artinya BELUM DIEVALUASI,
# bukan otomatis "sesuai". Jangan kasih default 'sesuai' di sini.
gdf_map['kategori_mismatch'] = gdf_map['nama_standar'].map(
    lambda n: mismatch_map[n]['kategori'] if n in mismatch_map else None
)
gdf_map['residual_z'] = gdf_map['nama_standar'].map(lambda n: mismatch_map.get(n, {}).get('residualZ'))
gdf_map['nightlight'] = gdf_map['nama_standar'].map(lambda n: mismatch_map.get(n, {}).get('nightlight'))
gdf_map['prediksi_nightlight'] = gdf_map['nama_standar'].map(lambda n: mismatch_map.get(n, {}).get('nightlightPred'))

print('wilayah di peta tanpa nilai garis_kemiskinan:', gdf_map['garis_kemiskinan'].isna().sum(), '/', len(gdf_map))
print('wilayah yang dievaluasi mismatch:', gdf_map['kategori_mismatch'].notna().sum(), '/', len(gdf_map))

geojson_str = gdf_map.to_json()
print('ukuran geojson (KB):', round(len(geojson_str) / 1024, 1))

vc = gdf_map['kategori_mismatch'].value_counts()
stats = {
    'total_wilayah': int(len(gdf_map)),
    'terisi': int(gdf_map['garis_kemiskinan'].notna().sum()),
    'dievaluasi': int(gdf_map['kategori_mismatch'].notna().sum()),
    'sesuai': int(vc.get('sesuai', 0)),
    'under_estimate': int(vc.get('under_estimate', 0)),
    'over_estimate': int(vc.get('over_estimate', 0)),
}
print(stats)

top_mismatch = (
    gdf_map[gdf_map['residual_z'].notna()]
    .assign(abs_z=lambda d: d['residual_z'].abs())
    .sort_values('abs_z', ascending=False)
    .head(10)[['nama_standar', 'provinsi', 'residual_z', 'kategori_mismatch']]
    .to_dict('records')
)

stats_json = json.dumps(stats, ensure_ascii=False)
methods_json = json.dumps(methods, ensure_ascii=False)
top_mismatch_json = json.dumps(top_mismatch, ensure_ascii=False)


HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Lumen</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
:root{
  --text-primary:#1a1a1a; --text-secondary:#4d4d4d; --text-tertiary:#8a8a8a; --text-quaternary:#b8b8b8;
  --surface-raised:#ffffff; --surface-muted:#f2f3f5; --surface-strong:#ececf1;
  --border:#e4e4ea; --danger:#e5484d; --positive:#30a46c; --warning:#f5a524; --chart1:#3b82f6;
  --map-water:#cfe3f0;
}
@media (prefers-color-scheme: dark){
  :root{
    --text-primary:#f0f0f0; --text-secondary:#c4c4c4; --text-tertiary:#8f8f8f; --text-quaternary:#5f5f5f;
    --surface-muted:#17171a; --surface-raised:#1c1c20; --surface-strong:#2b2b31; --border:#35353c; --chart1:#5b9bf7;
  }
}
*{box-sizing:border-box}
body{margin:0;background:var(--surface-muted);font-family:-apple-system,"Segoe UI",Roboto,sans-serif;color:var(--text-primary)}
.app{display:flex;flex-direction:column;gap:16px;padding:20px;min-height:100vh}
header{display:flex;justify-content:space-between;align-items:flex-start;gap:12px}
h1{font-size:20px;margin:0}
.sub{font-size:13px;color:var(--text-tertiary);margin:4px 0 0}
.stage{position:relative;border-radius:16px;overflow:hidden;border:1px solid var(--border);background:var(--surface-raised);height:min(72vh,640px)}
/* warna air FIXED, gak ikut ganti pas sistem dark/light mode berubah */
#map{width:100%;height:100%;background:var(--map-water) !important}
.leaflet-container{background:var(--map-water) !important;font-family:inherit}
.floatstats{position:absolute;top:16px;right:16px;display:flex;flex-direction:column;gap:8px;z-index:500}
.fstat{display:flex;align-items:center;gap:10px;background:var(--surface-raised);border:1px solid var(--border);border-radius:12px;padding:8px 14px;box-shadow:0 6px 20px rgba(0,0,0,.12);min-width:180px}
.fstat .dot{width:9px;height:9px;border-radius:50%;flex:none}
.fstat.ok .dot{background:var(--positive)} .fstat.warn .dot{background:var(--warning)} .fstat.bad .dot{background:var(--danger)}
.fstat b{font-size:18px;font-variant-numeric:tabular-nums}
.fstat span{font-size:11px;color:var(--text-tertiary);display:block}
.floatdetail{position:absolute;bottom:16px;right:16px;width:280px;background:var(--surface-raised);border:1px solid var(--border);border-radius:14px;padding:16px;box-shadow:0 10px 30px rgba(0,0,0,.16);z-index:500}
.floatdetail h3{margin:0;font-size:15px}
.floatdetail .prov{font-size:12px;color:var(--text-tertiary);margin:2px 0 10px}
.bignum{font-size:24px;font-weight:600;font-variant-numeric:tabular-nums;line-height:1.15}
.bignum small{display:block;font-size:11px;color:var(--text-tertiary);font-weight:400;margin-top:3px}
.rows{margin-top:10px;border-top:1px solid var(--border)}
.row{display:flex;justify-content:space-between;padding:6px 0;border-bottom:1px solid var(--border);font-size:12.5px;gap:8px}
.row:last-child{border-bottom:0}
.row .k{color:var(--text-secondary)}
.row .v{font-variant-numeric:tabular-nums;font-weight:600;text-align:right}
.pill{font-size:10.5px;padding:1px 8px;border-radius:99px;font-weight:600;white-space:nowrap}
.pill.ok{background:rgba(48,164,108,.14);color:var(--positive)}
.pill.warn{background:rgba(245,165,36,.16);color:var(--warning)}
.pill.bad{background:rgba(229,72,77,.13);color:var(--danger)}
.legend{position:absolute;bottom:16px;left:16px;background:var(--surface-raised);border:1px solid var(--border);border-radius:10px;padding:8px 12px;font-size:11.5px;color:var(--text-secondary);z-index:500;display:flex;gap:12px}
.legend .li{display:flex;align-items:center;gap:6px}
.legend .sw{width:10px;height:10px;border-radius:3px}
.charts{display:grid;grid-template-columns:1fr 1fr;gap:16px}
.card{background:var(--surface-raised);border:1px solid var(--border);border-radius:14px;padding:16px}
.chead{display:flex;justify-content:space-between;align-items:center;margin-bottom:10px}
.chead b{font-size:14px}
.mbar{margin-top:10px}
.mbar label{display:flex;justify-content:space-between;font-size:12px;color:var(--text-secondary);margin-bottom:4px}
.mbar label span{color:var(--text-primary);font-weight:600;font-variant-numeric:tabular-nums}
.track{height:7px;border-radius:99px;background:var(--surface-muted);overflow:hidden}
.fill{height:100%;border-radius:99px;background:var(--chart1);width:0;transition:width .8s}
.fill.dim{background:var(--text-quaternary)}
.mmrow{display:flex;justify-content:space-between;align-items:center;padding:7px 0;border-bottom:1px solid var(--border);font-size:12.5px}
.mmrow:last-child{border-bottom:0}
@media(max-width:900px){.charts{grid-template-columns:1fr}.floatdetail{width:220px}}
</style>
</head>
<body>
<div class="app">
  <header>
    <div>
      <h1>Cahaya vs Kenyataan</h1>
      <p class="sub">Estimasi garis kemiskinan · __DIEVALUASI__ dari __TOTAL_WILAYAH__ kabupaten/kota dievaluasi · nightlight vs realita</p>
    </div>
  </header>
  <div class="stage">
    <div id="map"></div>
    <div class="floatstats">
      <div class="fstat ok"><div class="dot"></div><div><b>__SESUAI__</b><span>estimasi sesuai (|z|≤1)</span></div></div>
      <div class="fstat bad"><div class="dot"></div><div><b>__UNDER__</b><span>under-estimate</span></div></div>
      <div class="fstat warn"><div class="dot"></div><div><b>__OVER__</b><span>over-estimate</span></div></div>
    </div>
    <div class="legend">
      <div class="li"><span class="sw" style="background:var(--positive)"></span>sesuai</div>
      <div class="li"><span class="sw" style="background:var(--danger)"></span>under-estimate</div>
      <div class="li"><span class="sw" style="background:var(--warning)"></span>over-estimate</div>
      <div class="li"><span class="sw" style="background:var(--text-quaternary)"></span>tidak ada data</div>
    </div>
    <div class="floatdetail" id="detail">
      <h3>Klik wilayah di peta</h3>
      <p class="prov">untuk lihat detail estimasi</p>
    </div>
  </div>
  <div class="charts">
    <div class="card">
      <div class="chead"><b>Perbandingan metode — RRMSE</b></div>
      <div id="methodsBars"></div>
    </div>
    <div class="card">
      <div class="chead"><b>10 wilayah mismatch paling ekstrem</b></div>
      <div id="topMismatch"></div>
    </div>
  </div>
</div>
<script>
const GEOJSON = __GEOJSON_DATA__;
const STATS = __STATS_DATA__;
const METHODS = __METHODS_DATA__;
const TOP_MISMATCH = __TOP_MISMATCH_DATA__;

document.querySelector('.floatstats .ok b').textContent = STATS.sesuai;
document.querySelector('.floatstats .bad b').textContent = STATS.under_estimate;
document.querySelector('.floatstats .warn b').textContent = STATS.over_estimate;

const fmtRp = n => n == null ? '—' : 'Rp ' + Math.round(n).toString().replace(/\B(?=(\d{3})+(?!\d))/g,'.');
const fmt1 = n => n == null ? '—' : n.toFixed(1).replace('.', ',');
const WARNA = { sesuai: 'var(--positive)', under_estimate: 'var(--danger)', over_estimate: 'var(--warning)' };

const map = L.map('map', { zoomControl: true, attributionControl: false }).setView([-2.5, 118], 5);

function warnaWilayah(f) {
  const p = f.properties;
  if (p.kategori_mismatch == null) return 'var(--text-quaternary)';
  return WARNA[p.kategori_mismatch] || 'var(--text-quaternary)';
}

function tampilkanDetail(p) {
  const zPill = p.residual_z == null ? ''
    : `<span class="pill ${Math.abs(p.residual_z) > 2.5 ? 'bad' : 'warn'}">${p.residual_z > 0 ? '+' : ''}${fmt1(p.residual_z)}σ</span>`;
  document.getElementById('detail').innerHTML = `
    <h3>${p.nama_standar}</h3>
    <p class="prov">${p.provinsi || '—'}</p>
    <div class="bignum">${fmtRp(p.garis_kemiskinan)}<small>garis kemiskinan · sumber: ${p.sumber}</small></div>
    <div class="rows">
      <div class="row"><span class="k">Nightlight</span><span class="v">${p.nightlight != null ? p.nightlight.toFixed(3) : '—'}</span></div>
      <div class="row"><span class="k">Prediksi nightlight-only</span><span class="v">${fmtRp(p.prediksi_nightlight)}</span></div>
      <div class="row"><span class="k">Residual z-score</span><span class="v">${zPill || '—'}</span></div>
      <div class="row"><span class="k">Kategori</span><span class="v">${p.kategori_mismatch || 'tidak dievaluasi'}</span></div>
    </div>`;
}

const geoLayer = L.geoJSON(GEOJSON, {
  style: f => ({ fillColor: warnaWilayah(f), fillOpacity: f.properties.garis_kemiskinan == null ? 0.15 : 0.55, color: 'var(--border)', weight: 0.6 }),
  onEachFeature: (f, layer) => {
    layer.on('click', () => tampilkanDetail(f.properties));
    layer.on('mouseover', e => e.target.setStyle({ weight: 2, color: '#333' }));
    layer.on('mouseout', e => e.target.setStyle({ weight: 0.6, color: 'var(--border)' }));
    layer.bindTooltip(f.properties.nama_standar, { sticky: true });
  }
}).addTo(map);
map.fitBounds(geoLayer.getBounds());

const maxRrmse = Math.max(...Object.values(METHODS).map(m => m.rrmse_pct));
document.getElementById('methodsBars').innerHTML = Object.entries(METHODS).map(([key, m]) => `
  <div class="mbar"><label>${m.deskripsi} <span>${m.rrmse_pct.toString().replace('.', ',')}%</span></label>
  <div class="track"><div class="fill ${key === 'sae_multivariat' ? '' : 'dim'}" style="width:${(m.rrmse_pct / maxRrmse * 100).toFixed(0)}%"></div></div></div>`).join('');

document.getElementById('topMismatch').innerHTML = TOP_MISMATCH.map(w => `
  <div class="mmrow"><span>${w.nama_standar}</span>
  <span class="pill ${w.kategori_mismatch === 'under_estimate' ? 'bad' : 'warn'}">${w.residual_z > 0 ? '+' : ''}${fmt1(w.residual_z)}σ</span></div>`).join('');
</script>
</body>
</html>"""

html_final = (HTML_TEMPLATE
    .replace('__GEOJSON_DATA__', geojson_str)
    .replace('__STATS_DATA__', stats_json)
    .replace('__METHODS_DATA__', methods_json)
    .replace('__TOP_MISMATCH_DATA__', top_mismatch_json)
    .replace('__TOTAL_WILAYAH__', str(stats['total_wilayah']))
    .replace('__DIEVALUASI__', str(stats['dievaluasi']))
    .replace('__SESUAI__', str(stats['sesuai']))
    .replace('__UNDER__', str(stats['under_estimate']))
    .replace('__OVER__', str(stats['over_estimate']))
)

OUT_PATH = f'{OUTPUT_FOLDER}/dashboard_cahaya_vs_kenyataan.html'
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    f.write(html_final)

print('\nSELESAI. Ukuran file (MB):', round(os.path.getsize(OUT_PATH) / 1e6, 2))
print('Tersimpan di:', OUT_PATH)
print('Download dari Drive terus buka langsung di browser (double-click).')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.9 MB/s eta 0:00:00
Mounted at /content/drive
geometri wilayah: 502 | gagal dicocokkan: 7
panel_kabkota_final: (512, 63)
wilayah di data_dashboard_nightlight.json: 492
wilayah di peta tanpa nilai garis_kemiskinan: 5 / 494
wilayah yang dievaluasi mismatch: 488 / 494
ukuran geojson (KB): 2680.8
{'total_wilayah': 494, 'terisi': 489, 'dievaluasi': 488, 'sesuai': 358, 'under_estimate': 73, 'over_estimate': 57}

SELESAI. Ukuran file (MB): 2.76
Tersimpan di: /content/drive/MyDrive/ARSEN/hasil_standarisasi/dashboard_cahaya_vs_kenyataan.html
Download dari Drive terus buka langsung di browser (double-click).
